In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px
import plotly.graph_objects as go
zema = ZemaManager()


In [0]:
start_date = datetime(2018, 1, 1)
end_date = datetime(2030, 1, 1)

corn_factor_bu_to_mt=0.3937

# CORN
## SPOT CBOT EVOLUTION 

In [0]:
cbot_corn= zema.get_curve(curve="P-FUTURE-CBOT-INPUT-CORN-USDc-BU", period=f"{start_date}::{end_date}")
cbot_corn=cbot_corn[cbot_corn['observation']=='Settle']
cbot_corn=cbot_corn[['date','value','contract_year','contract_month']]
cbot_corn['day']=cbot_corn['date'].dt.day
cbot_corn['month']=cbot_corn['date'].dt.month
# CBOT month codes (if you want codes like Z25, H26, ...)
month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}

def get_spot_contract(row):
    d = row['date']
    y = d.year
    m = d.month
    day = d.day

    # Sep 20 -> Dec 12  => Dec of current year
    if (m == 9 and day >= 13) or (m in [10, 11]) or (m == 12 and day <= 12):
        return (y, 12)

    # Dec 20 -> Dec 31 => March of NEXT year (roll happens at year end)
    if m == 12 and day >= 13:
        return (y + 1, 3)

    # Jan 1 -> Mar 12  => March of CURRENT year
    if (m in [1, 2]) or (m == 3 and day <= 12):
        return (y, 3)

    # Mar 20 -> May 12 => May of current year
    if (m == 3 and day >= 13) or (m == 4) or (m == 5 and day <= 12):
        return (y, 5)

    # May 13 -> Jul 12 => July of current year
    if (m == 5 and day >= 13) or (m == 6) or (m == 7 and day <= 12):
        return (y, 7)

    # Jul 13 -> Sep 12 => September of current year
    if (m == 7 and day >= 13) or (m == 8) or (m == 9 and day <= 12):
        return (y, 9)

    # safety fallback (shouldn't be hit with your rules)
    return (y, m)


# Apply to your cbot_corn dataframe
cbot_corn['spot_year'], cbot_corn['spot_month'] = zip(*cbot_corn.apply(get_spot_contract, axis=1))
cbot_corn['spot_code'] = cbot_corn.apply(lambda r: month_codes[r['spot_month']] + str(r['spot_year'])[-2:], axis=1)

# Filter to keep only the contract rows that correspond to the rolling spot
spot_corn = cbot_corn[
    (cbot_corn['contract_year'] == cbot_corn['spot_year']) &
    (cbot_corn['contract_month'] == cbot_corn['spot_month'])
].copy()

spot_corn = spot_corn.sort_values('date')



In [0]:
cbot_spot = go.Figure()

# Add the rolling spot contract line
cbot_spot.add_trace(go.Scatter(
    x=spot_corn['date'],
    y=spot_corn['value'],
    mode='lines',
    name="CBOT Corn Spot",
    line=dict(color="blue", width=2)
))

# Add markers when the contract changes
roll_dates = spot_corn.loc[spot_corn['spot_code'].shift() != spot_corn['spot_code'], 'date']
roll_labels = spot_corn.loc[spot_corn['spot_code'].shift() != spot_corn['spot_code'], 'spot_code']

for d, code in zip(roll_dates, roll_labels):
    cbot_spot.add_vline(x=d, line_dash="dash", line_color="gray")
    cbot_spot.add_annotation(
        x=d, y=spot_corn['value'].max(),
        text=f"Roll → {code}",
        showarrow=False,
        yshift=20,
        font=dict(size=10, color="gray")
    )

# Layout settings
cbot_spot.update_layout(
    title="CBOT Corn Rolling Spot Contract",
    xaxis_title="Date",
    yaxis_title="Price (USDc/bu)",
    hovermode="x unified",
    template="plotly_white"
)

cbot_spot.show()
cbot_spot_html = cbot_spot.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
df_prices=cbot_corn[['date', 'value', 'contract_year', 'contract_month', 'day', 'month',
       'spot_year', 'spot_month']]

month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}

# ---------------------------
# Assign virtual date for seasonal alignment (like spreads)
# ---------------------------
def assign_virtual_date_with_history(row):
    d = row['date']
    cm = row['contract_month']
    year = row['contract_year']  # the contract year
    y0 = 2000  # base year for plotting

    def safe_date(y, m, day):
        try:
            return pd.Timestamp(year=y, month=m, day=day)
        except ValueError:
            if m == 2 and day == 29:
                return pd.Timestamp(year=y, month=2, day=28)
            return pd.NaT

    # Define seasonal start/end for each contract month
    if cm == 3:  # H
        season_start = pd.Timestamp(year=year-1, month=4, day=1)
        season_end   = pd.Timestamp(year=year, month=3, day=12)
    elif cm == 5:  # K
        season_start = pd.Timestamp(year=year-1, month=6, day=1)
        season_end   = pd.Timestamp(year=year, month=5, day=12)
    elif cm == 7:  # N
        season_start = pd.Timestamp(year=year-1, month=8, day=1)
        season_end   = pd.Timestamp(year=year, month=7, day=12)
    elif cm == 9:  # U
        season_start = pd.Timestamp(year=year-1, month=11, day=1)
        season_end   = pd.Timestamp(year=year, month=9, day=30)
    elif cm == 12:  # Z
        season_start = pd.Timestamp(year=year, month=1, day=1)
        season_end   = pd.Timestamp(year=year, month=12, day=12)
    else:
        return pd.NaT

    # Compute virtual year offset
    if d < season_start:
        virtual_year = y0 - (season_start.year - d.year)
    else:
        virtual_year = y0 + (d.year - season_start.year)

    return safe_date(virtual_year, d.month, d.day)

# Apply to all prices
df_prices["virtual_date"] = df_prices.apply(assign_virtual_date_with_history, axis=1)
df_prices = df_prices.dropna(subset=["virtual_date"])

# ---------------------------
# Plot
# ---------------------------
cbot_contract_ev = go.Figure()
contract_months = sorted(df_prices["contract_month"].unique())
colors = px.colors.qualitative.Plotly

for cm in contract_months:
    df_sub = df_prices[df_prices["contract_month"] == cm]
    years = sorted(df_sub["contract_year"].unique())
    season_color_map = {year: colors[i % len(colors)] for i, year in enumerate(years)}
    
    for year in years:
        df_year = df_sub[df_sub["contract_year"] == year].sort_values("virtual_date")
        cbot_contract_ev.add_trace(go.Scatter(
            x=df_year["virtual_date"],
            y=df_year["value"],
            mode="lines",
            name=f"{month_codes[cm]}{year}",
            line=dict(color=season_color_map[year], width=2)
        ))

# ---------------------------
# Buttons to filter by contract month
# ---------------------------
buttons = []
for cm in contract_months:
    visible = [f"{month_codes[cm]}" in trace.name for trace in cbot_contract_ev.data]
    buttons.append(dict(label=month_codes[cm], method="update", args=[{"visible": visible}]))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(cbot_contract_ev.data)}]))

# ---------------------------
# Layout
# ---------------------------
cbot_contract_ev.update_layout(
    title=dict(
        text="CBOT Corn Prices by Contract Month<br><sup>Seasonal Alignment</sup>",
        x=0.5, xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Price (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b"),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

cbot_contract_ev.show()

# Export HTML if needed
cbot_contract_ev_html = cbot_contract_ev.to_html(include_plotlyjs='cdn', full_html=True)


### SPREADS

In [0]:

# Contract month codes
month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}
spread_pairs = [(12, 3), (3, 5), (5, 7), (7, 9), (9, 12)]  # (near, far)

# Create contract_date to align contracts
cbot_corn['contract_date'] = pd.to_datetime(
    dict(year=cbot_corn['contract_year'], month=cbot_corn['contract_month'], day=1)
)

# Pivot contracts per date (columns = contract code)
pivot = cbot_corn.pivot_table(
    index="date",
    columns=["contract_year", "contract_month"],
    values="value"
)

# Compute spreads
spreads = pd.DataFrame(index=pivot.index)

for near, far in spread_pairs:
    # For each year, match near vs far contracts
    for year in cbot_corn['contract_year'].unique():
        try:
            near_price = pivot[(year, near)]
            far_price  = pivot[(year + (1 if near == 12 and far == 3 else 0), far)]
            spread_name = f"{month_codes[near]}{str(year)[-2:]}-{month_codes[far]}{str(year + (1 if near == 12 and far == 3 else 0))[-2:]}"
            spreads[spread_name] = near_price - far_price
        except KeyError:
            continue

all_spreads = go.Figure()

# --- Build traces
for col in spreads.columns:
    all_spreads.add_trace(go.Scatter(
        x=spreads.index,
        y=spreads[col],
        mode='lines',
        name=col
    ))

# --- Build season list (e.g. "19", "20", …)
# Extract last 2 digits of year from column names
season_list = sorted(set([c[-2:] for c in spreads.columns]))

# --- Create buttons
buttons = []
for season in season_list:
    visible = [season in trace.name for trace in all_spreads.data]
    buttons.append(dict(
        label=f"20{season}",
        method="update",
        args=[{"visible": visible}]
    ))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True] * len(all_spreads.data)}]))

# --- Update layout
all_spreads.update_layout(
    title="CBOT Corn Calendar Spreads",
    xaxis_title="Date",
    yaxis_title="Spread (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

all_spreads.show()

all_spreads_html = all_spreads.to_html(include_plotlyjs='cdn', full_html=True)

In [0]:
# ---------------------------
# Compute spreads (assuming 'cbot_corn' pivoted as before)
# ---------------------------
month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}
spread_pairs = [(12, 3), (3, 5), (5, 7), (7, 9), (9, 12)]

pivot = cbot_corn.pivot_table(
    index="date",
    columns=["contract_year", "contract_month"],
    values="value"
)

spread_list = []
for near, far in spread_pairs:
    for year in cbot_corn['contract_year'].unique():
        try:
            near_price = pivot[(year, near)]
            far_price  = pivot[(year + (1 if near == 12 and far == 3 else 0), far)]
            spread_name = f"{month_codes[near]}{str(year)[-2:]}-{month_codes[far]}{str(year + (1 if near == 12 and far == 3 else 0))[-2:]}"
            tmp = pd.DataFrame({
                "date": near_price.index,
                "spread": near_price - far_price,
                "spread_type": f"{month_codes[near]}{month_codes[far]}",
                "season": year
            })
            spread_list.append(tmp)
        except KeyError:
            continue

spreads = pd.concat(spread_list).dropna()

# ---------------------------
# Filter last 9 months per spread/season
# ---------------------------
def keep_all_history(df):
    df_list = []
    for stype in df['spread_type'].unique():
        df_sub = df[df['spread_type'] == stype]
        for season in df_sub['season'].unique():
            tmp = df_sub[df_sub['season'] == season].copy()
            if tmp.empty:
                continue

            # Define seasonal start and end (for reference, optional)
            if stype == "ZH":
                season_start = pd.Timestamp(season-1, 4, 1)
                season_end   = pd.Timestamp(season, 3, 31)
            elif stype == "HK":
                season_start = pd.Timestamp(season-1, 6, 1)
                season_end   = pd.Timestamp(season, 5, 31)
            elif stype == "KN":
                season_start = pd.Timestamp(season-1, 8, 1)
                season_end   = pd.Timestamp(season, 7, 31)
            elif stype == "NU":
                season_start = pd.Timestamp(season-1, 11, 1)
                season_end   = pd.Timestamp(season, 9, 30)
            elif stype == "UZ":
                season_start = pd.Timestamp(season, 1, 1)
                season_end   = pd.Timestamp(season, 12, 31)

            # Keep all data (no cutoff)
            tmp = tmp[(tmp['date'] >= tmp['date'].min()) & (tmp['date'] <= tmp['date'].max())]
            df_list.append(tmp)

    return pd.concat(df_list)

spreads = keep_all_history(spreads)

# ---------------------------
# Assign virtual date for seasonal alignment (safe with leap years)
# ---------------------------
def assign_virtual_date_with_history(row):
    d = row['date']
    stype = row['spread_type']
    season = row['season']  # the far contract year
    y0 = 2000  # base year for plotting
    
    def safe_date(y, m, day):
        try:
            return pd.Timestamp(year=y, month=m, day=day)
        except ValueError:
            if m == 2 and day == 29:
                return pd.Timestamp(year=y, month=2, day=28)
            return pd.NaT

    # Define natural seasonal start/end
    if stype == "ZH":
        season_start = pd.Timestamp(season-1, 4, 1)
        season_end   = pd.Timestamp(season, 3, 31)
    elif stype == "HK":
        season_start = pd.Timestamp(season-1, 6, 1)
        season_end   = pd.Timestamp(season, 5, 31)
    elif stype == "KN":
        season_start = pd.Timestamp(season-1, 8, 1)
        season_end   = pd.Timestamp(season, 7, 31)
    elif stype == "NU":
        season_start = pd.Timestamp(season-1, 11, 1)
        season_end   = pd.Timestamp(season, 9, 30)
    elif stype == "UZ":
        season_start = pd.Timestamp(season, 1, 1)
        season_end   = pd.Timestamp(season, 12, 31)
    else:
        return pd.NaT

    # Compute virtual year offset
    if d < season_start:
        virtual_year = y0 - (season_start.year - d.year)  # negative offset
    else:
        virtual_year = y0 + (d.year - season_start.year)  # 0 or positive

    return safe_date(virtual_year, d.month, d.day)

# Apply to all spreads
spreads["virtual_date"] = spreads.apply(assign_virtual_date_with_history, axis=1)
spreads = spreads.dropna(subset=["virtual_date"])

# ---------------------------
# Plot
# ---------------------------
cbot_cal_spread = go.Figure()
spread_types = spreads["spread_type"].unique()
colors = px.colors.qualitative.Plotly  # color palette

for stype in spread_types:
    df_sub = spreads[spreads["spread_type"] == stype]
    seasons = sorted(df_sub["season"].unique())
    season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(seasons)}
    
    for season in seasons:
        season_data = df_sub[df_sub["season"] == season]
        season_data = season_data.sort_values("virtual_date")

        cbot_cal_spread.add_trace(go.Scatter(
            x=season_data["virtual_date"],
            y=season_data["spread"],
            mode="lines",
            name=f"{stype} {season}",
            line=dict(color=season_color_map[season], width=2)
        ))

# ---------------------------
# Buttons to filter by spread type
# ---------------------------
buttons = []
for stype in spread_types:
    visible = [stype in trace.name for trace in cbot_cal_spread.data]
    buttons.append(dict(label=stype, method="update", args=[{"visible": visible}]))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(cbot_cal_spread.data)}]))

# ---------------------------
# Layout
# ---------------------------
cbot_cal_spread.update_layout(
    title=dict(
        text="CBOT Corn Calendar Spreads<br><sup>From first contract's quote to expiry</sup>",
        x=0.5,  # center
        xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Spread (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b"),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

cbot_cal_spread.show()


cbot_cal_spread_html = cbot_cal_spread.to_html(include_plotlyjs='cdn', full_html=True)

## FOB SPOT

In [0]:
import pandas as pd

def get_spot_curve(curve_name: str, start_date: str, end_date: str):
    """
    Fetches a curve from ZEMA and processes it to return the spot curve DataFrame.
    
    Parameters
    ----------
    curve_name : str
        The curve name in ZEMA.
    start_date : str or datetime
        Start date for the query (e.g. '2020-01-01').
    end_date : str or datetime
        End date for the query (e.g. '2025-01-01').

    Returns
    -------
    pd.DataFrame
        DataFrame with ['date', 'value', 'contract_year', 'contract_month',
        'virtual_date', 'contract_date'] for the spot curve.
    """
    
    # Fetch curve
    df = zema.get_curve(curve=curve_name, period=f"{start_date}::{end_date}")
    
    # Keep only "Last" observations
    df = df[df['observation'] == 'Last']
    
    # Keep relevant columns
    df = df[['date', 'value', 'contract_year', 'contract_month']].copy()
    
    # Extract calendar parts
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    
    # Create "virtual date" (aligning all years to 2000 for seasonality)
    df['virtual_date'] = pd.to_datetime(
        {'year': 2000, 'month': df['month'], 'day': df['day']},
        errors='coerce'
    )
    
    # Drop invalid dates
    df = df.dropna(subset=['virtual_date'])
    
    # Ensure datetime types
    df['date'] = pd.to_datetime(df['date'])
    df['virtual_date'] = pd.to_datetime(df['virtual_date'])
    
    # Create contract date (first day of contract month/year)
    df['contract_date'] = pd.to_datetime(dict(
        year=df['contract_year'],
        month=df['contract_month'],
        day=1
    ))
    
    # For each real date, pick the earliest available contract
    spot_df = (
        df.sort_values(['date', 'contract_date'])
          .groupby('date')
          .first()
          .reset_index()
    )
    
    return spot_df



def plot_flat_premium(flat_df=None, prem_df=None, title="Flat and Premium", width=1800, height=700, colors=None):
    """
    Plots Flat (FOB) and Premium values by season with interactive filtering.
    Works even if only one of flat_df or prem_df is provided.
    
    Parameters
    ----------
    flat_df : pd.DataFrame, optional
        DataFrame containing 'virtual_date', 'flat', and 'contract_year'.
    prem_df : pd.DataFrame, optional
        DataFrame containing 'virtual_date', 'premium', and 'contract_year'.
    title : str, optional
        Plot title (default: "Flat and Premium").
    width : int, optional
        Plot width in pixels (default: 1800).
    height : int, optional
        Plot height in pixels (default: 700).
    colors : list, optional
        List of colors to cycle through. Defaults to Plotly qualitative palette.
    """

    # Default colors
    if colors is None:
        colors = px.colors.qualitative.Plotly

    # Initialize empty figure
    fig = go.Figure()

    # Collect season list from both dfs
    season_list = set()
    if flat_df is not None and not flat_df.empty:
        flat_df = flat_df.copy()
        flat_df['season'] = flat_df['contract_year'].astype(str)
        season_list.update(flat_df['season'].unique())
    if prem_df is not None and not prem_df.empty:
        prem_df = prem_df.copy()
        prem_df['season'] = prem_df['contract_year'].astype(str)
        season_list.update(prem_df['season'].unique())

    season_list = sorted(season_list)
    season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(season_list)}

    # Add FOB traces if flat_df exists
    if flat_df is not None and not flat_df.empty:
        for season in season_list:
            season_data = flat_df[flat_df['season'] == season].sort_values('virtual_date')
            if not season_data.empty:
                fig.add_trace(go.Scatter(
                    x=season_data['virtual_date'],
                    y=season_data['flat'],
                    mode='lines',
                    name=f"FOB {season}",
                    line=dict(color=season_color_map[season], width=2, dash='solid'),
                    yaxis="y1"
                ))

    # Add Premium traces if prem_df exists
    if prem_df is not None and not prem_df.empty:
        for season in season_list:
            season_data = prem_df[prem_df['season'] == season].sort_values('virtual_date')
            if not season_data.empty:
                fig.add_trace(go.Scatter(
                    x=season_data['virtual_date'],
                    y=season_data['premium'],
                    mode='lines',
                    name=f"Premium {season}",
                    line=dict(color=season_color_map[season], width=2, dash='dash'),
                    yaxis="y2"
                ))

    # Layout with dual y-axes
    fig.update_layout(
        title=title,
        xaxis=dict(
            tickformat='%b',
            dtick="M1",
            hoverformat='%d-%b'
        ),
        yaxis=dict(title='FOB Spot Price'),
        yaxis2=dict(
            title='Premium',
            overlaying='y',
            side='right',
            showgrid=False
        ),
        template='plotly_white',
        height=height,
        width=width,
        hovermode='x unified'
    )

    # Add Buttons for filtering by Season
    if season_list:
        buttons = []
        for season in season_list:
            visible = [season in trace.name for trace in fig.data]
            buttons.append(dict(
                label=season,
                method="update",
                args=[{"visible": visible}]
            ))
        buttons.insert(0, dict(
            label="ALL",
            method="update",
            args=[{"visible": [True] * len(fig.data)}]
        ))

        fig.update_layout(
            updatemenus=[dict(
                type="buttons",
                direction="left",
                x=0,
                y=1.05,
                xanchor="left",
                yanchor="top",
                buttons=buttons,
                showactive=True
            )]
        )

    return fig


### USG

In [0]:
corn_spot_usg_flat=get_spot_curve('P-CASH-LDC-CALC-FLAT-CORN-US-FOB-FBV-CGF-YELLOW-USDc-Bu',start_date,end_date)
corn_spot_usg_flat['value']=corn_spot_usg_flat['value']*corn_factor_bu_to_mt
corn_spot_usg_flat.rename(columns={'value':'flat'},inplace=True)
corn_spot_usg_flat['season'] = "FOB " + corn_spot_usg_flat['year'].astype(str)

corn_spot_usg_prem=get_spot_curve('P-CASH-LDC-CALC-PREMIUM-CORN-US-FOB-FBV-CGF-YELLOW-USDc-Bu',start_date,end_date)
corn_spot_usg_prem.rename(columns={'value':'premium'},inplace=True)
corn_spot_usg_prem['season'] = "Premium " + corn_spot_usg_prem['year'].astype(str)

corn_usg_spot_chart = plot_flat_premium(corn_spot_usg_flat, corn_spot_usg_prem, title="CORN USG Spot Flat and Premium")
corn_usg_spot_chart.show()

corn_usg_spot_chart_html = corn_usg_spot_chart.to_html(include_plotlyjs='cdn', full_html=True)


### UPBB

In [0]:
corn_spot_upbb_flat=get_spot_curve('P-CASH-LDC-CALC-FLAT-CORN-AR-FOB Up River Arg-USD-MT',start_date,end_date)
corn_spot_upbb_flat.rename(columns={'value':'flat'},inplace=True)
corn_spot_upbb_flat['season'] = "FOB " + corn_spot_upbb_flat['year'].astype(str)


corn_spot_upbb_prem=get_spot_curve('P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Up River Arg-USDc-Bu',start_date,end_date)
corn_spot_upbb_prem.rename(columns={'value':'premium'},inplace=True)
corn_spot_upbb_prem['season'] = "Premium " + corn_spot_upbb_prem['year'].astype(str)

corn_upr_spot_chart = plot_flat_premium(corn_spot_upbb_flat, corn_spot_upbb_prem, title="CORN UPR Spot Flat and Premium")
corn_upr_spot_chart.show()



corn_upr_spot_chart_html = corn_upr_spot_chart.to_html(include_plotlyjs='cdn', full_html=True)


### SANTOS

In [0]:
corn_spot_santos_flat=get_spot_curve('P-CASH-LDC-CALC-FLAT-CORN-BR-FOB Santos-USD-MT',start_date,end_date)
corn_spot_santos_flat.rename(columns={'value':'flat'},inplace=True)
corn_spot_santos_flat['season'] = "FOB " + corn_spot_santos_flat['year'].astype(str)

corn_spot_santos_prem=get_spot_curve('P-CASH-LDC-INPUT-PREMIUM-CORN-BR-FOB Santos-USDc-Bu',start_date,end_date)
corn_spot_santos_prem.rename(columns={'value':'premium'},inplace=True)
corn_spot_santos_prem['season'] = "Premium " + corn_spot_santos_prem['year'].astype(str)

corn_santos_spot_chart = plot_flat_premium(corn_spot_santos_flat, corn_spot_santos_prem, title="CORN Santos Spot Flat and Premium")
corn_santos_spot_chart.show()

corn_santos_spot_chart_html = corn_santos_spot_chart.to_html(include_plotlyjs='cdn', full_html=True)


### Odessa

In [0]:
corn_spot_odes_prem=get_spot_curve('P-CASH-LDC-INPUT-PREMIUM-CORN-UA-FOB ODESSA-USDc-BU',start_date,end_date)
corn_spot_odes_prem.rename(columns={'value':'premium'},inplace=True)

corn_spot_odes_prem['season'] = "Premium " + corn_spot_odes_prem['year'].astype(str)

corn_odessa_spot_chart = plot_flat_premium(prem_df= corn_spot_odes_prem, title="CORN Odessa Spot Premium")
corn_odessa_spot_chart.show()

corn_odessa_spot_chart_html = corn_odessa_spot_chart.to_html(include_plotlyjs='cdn', full_html=True)


## MATBA PREMIUMS

In [0]:
corn_matba= zema.get_curve(curve="P-FUTURE-MATBA-INPUT-CORN-USD-MT", period=f"{start_date}::{end_date}")
corn_matba=corn_matba[corn_matba['observation']=='Settle']
corn_matba=corn_matba[['date','value','contract_year','contract_month']]
corn_matba.rename(columns={'value':'matba'},inplace=True)

corn_cbot= zema.get_curve(curve="P-FUTURE-CBOT-INPUT-CORN-USDc-BU", period=f"{start_date}::{end_date}")
corn_cbot=corn_cbot[corn_cbot['observation']=='Settle']
corn_cbot=corn_cbot[['date','value','contract_year','contract_month']]
corn_cbot.rename(columns={'value':'cbot'},inplace=True)


corn_xchg=pd.merge(corn_cbot,corn_matba,on=['date','contract_year','contract_month'],how='inner')
corn_xchg = corn_xchg[(corn_xchg['matba'] != 0) & (corn_xchg['cbot'] != 0)]
corn_xchg['matba']=corn_xchg['matba']*(1/corn_factor_bu_to_mt)
corn_xchg['spread']=corn_xchg['matba']-corn_xchg['cbot']

corn_xchg


In [0]:
corn_matba= zema.get_curve(curve="P-FUTURE-MATBA-INPUT-CORN-USD-MT", period=f"{start_date}::{end_date}")
corn_matba=corn_matba[corn_matba['observation']=='Settle']
corn_matba=corn_matba[['date','value','contract_year','contract_month']]
corn_matba.rename(columns={'value':'matba'},inplace=True)

corn_cbot= zema.get_curve(curve="P-FUTURE-CBOT-INPUT-CORN-USDc-BU", period=f"{start_date}::{end_date}")
corn_cbot=corn_cbot[corn_cbot['observation']=='Settle']
corn_cbot=corn_cbot[['date','value','contract_year','contract_month']]
corn_cbot.rename(columns={'value':'cbot'},inplace=True)

# First, separate MATBA April contracts
matba_april = corn_matba[corn_matba['contract_month'] == 4].copy()
matba_other = corn_matba[corn_matba['contract_month'] != 4].copy()

# Shift April to align with May
matba_april['contract_month'] = 5  

# Recombine MATBA data
corn_matba_adj = pd.concat([matba_other, matba_april], ignore_index=True)

# Now merge with CBOT
corn_xchg = pd.merge(
    corn_cbot, 
    corn_matba_adj,
    on=['date', 'contract_year', 'contract_month'],
    how='inner'
)

# Clean spreads
corn_xchg = corn_xchg[(corn_xchg['matba'] != 0) & (corn_xchg['cbot'] != 0)]
corn_xchg['matba'] = corn_xchg['matba'] * (1 / corn_factor_bu_to_mt)
corn_xchg['spread'] = corn_xchg['matba'] - corn_xchg['cbot']
corn_xchg

In [0]:
def assign_virtual_date(row):
    d = row["date"]
    contract_year = row["contract_year"]
    y0 = 2000  # base year for alignment

    def safe_date(y, m, day):
        try:
            return pd.Timestamp(year=y, month=m, day=day)
        except ValueError:
            if m == 2 and day == 29:
                return pd.Timestamp(year=y, month=2, day=28)
            return pd.NaT

    # Offset between actual calendar and contract's expiry year
    year_offset = d.year - contract_year
    virtual_year = y0 + year_offset

    return safe_date(virtual_year, d.month, d.day)


### APRIL

In [0]:
corn_xchg_APR=corn_xchg[corn_xchg['contract_month']==5]
# Remove rows where contract_year == date.year and date between Apr 14 and May 31
mask = ~(
    (corn_xchg_APR['contract_year'] == corn_xchg_APR['date'].dt.year) &
    (corn_xchg_APR['date'].dt.month.isin([4, 5])) &
    ~((corn_xchg_APR['date'].dt.month == 4) & (corn_xchg_APR['date'].dt.day < 14))
)

corn_xchg_APR = corn_xchg_APR[mask].copy()

corn_xchg_APR['Season'] = 'APR'+ corn_xchg_APR['contract_year'].astype(str)


In [0]:
df = corn_xchg_APR.copy()  # has: date, spread, Season, contract_year, contract_month

df["virtual_date"] = df.apply(assign_virtual_date, axis=1)
df = df.dropna(subset=["virtual_date"])

# ---------------------------
# Plot spreads by season
# ---------------------------
chart_april_spread_corn = go.Figure()
seasons = sorted(df["Season"].unique())
colors = px.colors.qualitative.Plotly
season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(seasons)}

for season in seasons:
    season_data = df[df["Season"] == season].sort_values("virtual_date")
    chart_april_spread_corn.add_trace(go.Scatter(
        x=season_data["virtual_date"],
        y=season_data["spread"],
        mode="lines",
        name=season,
        line=dict(color=season_color_map[season], width=2)
    ))

# ---------------------------
# Buttons: one per season + ALL
# ---------------------------
buttons = []
for season in seasons:
    visible = [season in trace.name for trace in chart_april_spread_corn.data]
    buttons.append(dict(label=season, method="update", args=[{"visible": visible}]))

buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(chart_april_spread_corn.data)}]))

chart_april_spread_corn.update_layout(
    title=dict(
        text="Matba April vs CBOT May Spreads",
        x=0.5, xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="MATBA April – CBOT May (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b"),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

chart_april_spread_corn.show()

chart_april_spread_corn_html = chart_april_spread_corn.to_html(include_plotlyjs='cdn', full_html=True)

### JULY

In [0]:
corn_xchg_JUL=corn_xchg[corn_xchg['contract_month']==7]
corn_xchg_JUL['Season'] = 'JUL'+ corn_xchg_JUL['contract_year'].astype(str)


In [0]:
df = corn_xchg_JUL.copy()  # has: date, spread, Season, contract_year, contract_month

df["virtual_date"] = df.apply(assign_virtual_date, axis=1)
df = df.dropna(subset=["virtual_date"])

# ---------------------------
# Plot spreads by season
# ---------------------------
chart_july_spread_corn = go.Figure()
seasons = sorted(df["Season"].unique())
colors = px.colors.qualitative.Plotly
season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(seasons)}

for season in seasons:
    season_data = df[df["Season"] == season].sort_values("virtual_date")
    chart_july_spread_corn.add_trace(go.Scatter(
        x=season_data["virtual_date"],
        y=season_data["spread"],
        mode="lines",
        name=season,
        line=dict(color=season_color_map[season], width=2)
    ))

# ---------------------------
# Buttons: one per season + ALL
# ---------------------------
buttons = []
for season in seasons:
    visible = [season in trace.name for trace in chart_july_spread_corn.data]
    buttons.append(dict(label=season, method="update", args=[{"visible": visible}]))

buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(chart_july_spread_corn.data)}]))

chart_july_spread_corn.update_layout(
    title=dict(
        text="Matba vs CBOT July Spreads",
        x=0.5, xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Matba – CBOT (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b"),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

chart_july_spread_corn.show()
chart_july_spread_corn_html = chart_july_spread_corn.to_html(include_plotlyjs='cdn', full_html=True)


### DECEMBER


In [0]:
corn_xchg_DEC=corn_xchg[corn_xchg['contract_month']==12]
corn_xchg_DEC['Season'] = 'Dec'+ corn_xchg_DEC['contract_year'].astype(str)


In [0]:
df = corn_xchg_DEC.copy()
df["virtual_date"] = df.apply(assign_virtual_date, axis=1)
df = df.dropna(subset=["virtual_date"])

# ---------------------------
# Plot spreads by season
# ---------------------------
chart_dec_spread_corn = go.Figure()
seasons = sorted(df["Season"].unique())
colors = px.colors.qualitative.Plotly
season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(seasons)}

for season in seasons:
    season_data = df[df["Season"] == season].sort_values("virtual_date")
    chart_dec_spread_corn.add_trace(go.Scatter(
        x=season_data["virtual_date"],
        y=season_data["spread"],
        mode="lines",
        name=season,
        line=dict(color=season_color_map[season], width=2)
    ))

# ---------------------------
# Buttons: one per season + ALL
# ---------------------------
buttons = []
for season in seasons:
    visible = [season in trace.name for trace in chart_dec_spread_corn.data]
    buttons.append(dict(label=season, method="update", args=[{"visible": visible}]))

buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(chart_dec_spread_corn.data)}]))

chart_dec_spread_corn.update_layout(
    title=dict(
        text="Matba vs CBOT Dec Spreads",
        x=0.5, xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Dec Matba – CBOT (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b"),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

chart_dec_spread_corn.show()

chart_dec_spread_corn_html = chart_dec_spread_corn.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
corn_mkt_spnapshot  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Corn Market Snapshot Report</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Corn Market Snapshot Report</h1>
    <h2>CBOT</h2>
        <h2>CBOT Rolling spot price</h2>
            <div class="chart-container">{cbot_spot_html}</div>
        <h2>CBOT Contracts price evolution</h2>
            <div class="chart-container">{cbot_contract_ev_html}</div>

        <h2>CBOT CALENDAR SPREADS</h2>
            <h3>Historical spreads</h3>
            <div class="chart-container">{all_spreads_html}</div>

            <h3>Seasonal view</h3>
            <div class="chart-container">{cbot_cal_spread_html}</div>

    <h1>Main Markets Spot and Premium</h1>
        <h2>USG Spot price</h2>
            <div class="chart-container">{corn_usg_spot_chart_html}</div>
        <h2>UPR Spot price</h2>
            <div class="chart-container">{corn_upr_spot_chart_html}</div>
        <h2>Santos Spot price</h2>
            <div class="chart-container">{corn_santos_spot_chart_html}</div>
        <h2>Odessa Spot price</h2>
            <div class="chart-container">{corn_odessa_spot_chart_html}</div>
    <h1>MATBA Premium VS CBOT</h1>
        <h2>April MATBA VS May CBOT</h2>
            <div class="chart-container">{chart_april_spread_corn_html}</div>
        <h2>July MATBA VS CBOT</h2>
            <div class="chart-container">{chart_july_spread_corn_html}</div>
        <h2>December MATBA VS CBOT</h2>
            <div class="chart-container">{chart_dec_spread_corn_html}</div>
    

</body>
</html>
"""
corn_mkt_spnapshot_report_bytes = corn_mkt_spnapshot.encode("utf-8")

grains=['florian.girardi-ext@ldc.com','Roman.Avramishin@LDC.com','juan.garciafuentes@ldc.com','gonzalo.lascombes@ldc.com','juan.carnemolla@LDC.com','valentin.chiesa@ldc.com']
test_2=['florian.girardi-ext@ldc.com']
      
# Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=grains,
    subject=f'Corn Market Snapshot  {datetime.now().strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    mime_type="html",
    body='Please find the report attached',
    attachment={"corn_market_snapshot.html": corn_mkt_spnapshot_report_bytes}
)
